# GitSkills — data cleaning

Builds the cleaned dataset from the raw tables downloaded by `data_extraction.ipynb`.

- A **skill** is a distinct `SKILL.md` content (`file_sha`). It is **copied** when it appears in
  repositories of at least `MIN_OWNERS` different owners (we count owners, not repositories,
  so one user copying a skill into all their repos counts once).
- A repository is **eligible** if it is not a fork, not an aggregator (more than
  `AGGREGATOR_THRESHOLD` distinct skills) and its primary language is a programming language.

Outputs in `data/`: `repos_clean.csv`, `skills_copied.csv` and `adoptions.csv`
(one row per copied skill × repository).

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow.dataset as ds
import yaml

DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "gitskills" / "data"

AGGREGATOR_THRESHOLD = 100  # distinct skills per repository (top 1%)
MIN_OWNERS = 2

## 1. Load the raw tables

In [2]:
SKILL_COLUMNS = ["file_sha", "filename", "location_class", "frontmatter_valid",
                 "body_chars", "has_scripts", "has_references", "sibling_count"]

artifacts = (ds.dataset(RAW_DIR / "artifacts", format="parquet")
             .to_table(columns=["repo_full_name", "dedup_primary"] + SKILL_COLUMNS)
             .to_pandas())
repos = pd.read_parquet(RAW_DIR / "repos").rename(columns={"full_name": "repo_full_name"})

print(f"SKILL.md files:  {len(artifacts):,}")
print(f"distinct skills: {artifacts['file_sha'].nunique():,}")
print(f"repositories:    {len(repos):,}")

SKILL.md files:  3,797,117


distinct skills: 1,877,981
repositories:    282,200


## 2. Languages

Only repositories whose primary language is a programming language are eligible: a repository
written mostly in HTML, Markdown or YAML says little about its tech stack. Language types come from
[GitHub Linguist](https://github.com/github-linguist/linguist), the tool GitHub uses to detect a
repository's language (and so the source of the `language` column). Each language has a type:
`programming`, `markup`, `data` or `prose`. We use `languages.yml` from Linguist v9.6.0, the
release current when GitSkills was mined (July 2026).

In [3]:
with open(DATA_DIR / "linguist_languages_v9.6.0.yml") as f:
    linguist_types = {name: info["type"] for name, info in yaml.safe_load(f).items()}

repos["language"] = repos["language"].mask(repos["language"] == "")  # "" = not detected
repos["is_fork"] = repos["is_fork"].fillna(0).astype(int)
repos["language_kind"] = repos["language"].map(linguist_types).fillna("missing")

repos["language_kind"].value_counts()

language_kind
programming    230558
missing         30116
markup          21408
prose              85
data               33
Name: count, dtype: int64

## 3. Aggregators

Some repositories store thousands of skills collected from elsewhere (registries, mirrors,
marketplaces). Their language has nothing to do with the skills, so they are excluded from the
language analysis. About 1% of repositories hold more than half of all `SKILL.md` files.

In [4]:
per_repo = artifacts.groupby("repo_full_name").agg(
    n_skill_files=("file_sha", "size"), n_distinct_skills=("file_sha", "nunique"))
repos = repos.merge(per_repo, on="repo_full_name")

for t in [10, 50, 100, 500, 1000]:
    big = repos["n_distinct_skills"] > t
    share = repos.loc[big, "n_skill_files"].sum() / len(artifacts)
    print(f"> {t:>4} skills: {big.sum():>6,} repos ({big.mean():.1%}), {share:.1%} of SKILL.md files")

repos.nlargest(10, "n_distinct_skills")[["repo_full_name", "language", "stars", "n_distinct_skills"]]

>   10 skills: 39,756 repos (14.1%), 82.2% of SKILL.md files
>   50 skills:  6,377 repos (2.3%), 61.8% of SKILL.md files
>  100 skills:  2,829 repos (1.0%), 54.3% of SKILL.md files
>  500 skills:    560 repos (0.2%), 39.8% of SKILL.md files
> 1000 skills:    249 repos (0.1%), 33.2% of SKILL.md files


,repo_full_name,language,stars,n_distinct_skills
650,majiayu000/claude-skill-registry-data,HTML,16.0,89908
121403,NeuralBlitz/ncx,NaN,6.0,44737
124,gabrielmoreira/agent-skills-mirror,Python,11.0,42373
337,David-Li0406/meta-skill-evloving,NaN,0.0,41975
121436,NeuralBlitz/buggy,HTML,3.0,30308
15408,Klotzkette/claude-fuer-deutsches-recht,Python,1332.0,21955
4647,tools-only/X-Skills,Python,7.0,21466
266,dvcrn/openclaw-skills-marketplace,Python,23.0,21287
66917,majiayu000/claude-skill-registry,HTML,498.0,15316
6057,NeuralBlitz/Agent-Gateway,TypeScript,1.0,14267


In [5]:
repos["is_aggregator"] = (repos["n_distinct_skills"] > AGGREGATOR_THRESHOLD).astype(int)
repos["eligible"] = ((repos["is_aggregator"] == 0) & (repos["is_fork"] == 0)
                     & (repos["language_kind"] == "programming")).astype(int)
repos = repos.sort_values("repo_full_name").reset_index(drop=True)
repos["repo_id"] = repos.index

print(f"aggregators:           {repos['is_aggregator'].sum():,}")
print(f"forks:                 {repos['is_fork'].sum():,}")
print(f"eligible repositories: {repos['eligible'].sum():,}")

aggregators:           2,829
forks:                 3
eligible repositories: 228,037


## 4. Copied skills

One adoption per (skill, repository) pair; `n_paths` counts how many times the skill appears in
that repository. Skills found under a single owner are dropped.

In [6]:
adoptions = (artifacts.groupby(["file_sha", "repo_full_name"]).size().rename("n_paths").reset_index()
             .merge(repos[["repo_full_name", "repo_id", "owner", "eligible", "language"]],
                    on="repo_full_name"))

owners_per_skill = adoptions.groupby("file_sha")["owner"].nunique()
copied = owners_per_skill.index[owners_per_skill >= MIN_OWNERS]
adoptions = adoptions[adoptions["file_sha"].isin(copied)]

print(f"single-owner skills (dropped): {(owners_per_skill < MIN_OWNERS).sum():,}")
print(f"copied skills:                 {len(copied):,}")
print(f"adoptions of copied skills:    {len(adoptions):,}")

single-owner skills (dropped): 1,647,626
copied skills:                 230,355
adoptions of copied skills:    1,382,321


## 5. Skill table

Skill attributes come from the representative row of each hash (`dedup_primary == 1`), plus
adoption counts over all repositories and over eligible ones only.

In [7]:
skills = artifacts.loc[(artifacts["dedup_primary"] == 1) & artifacts["file_sha"].isin(copied),
                       SKILL_COLUMNS]

all_counts = adoptions.groupby("file_sha").agg(
    n_repos=("repo_id", "size"), n_owners=("owner", "nunique"))
eligible_counts = adoptions[adoptions["eligible"] == 1].groupby("file_sha").agg(
    n_repos_eligible=("repo_id", "size"), n_owners_eligible=("owner", "nunique"),
    n_languages=("language", "nunique"))

skills = skills.join(all_counts, on="file_sha").join(eligible_counts, on="file_sha")
skills[eligible_counts.columns] = skills[eligible_counts.columns].fillna(0).astype(int)
skills = skills.sort_values("file_sha").reset_index(drop=True)
skills.insert(0, "skill_id", skills.index)
adoptions = adoptions.merge(skills[["file_sha", "skill_id"]], on="file_sha")

print(f"copied skills with >= 2 eligible owners (RQ1): {(skills['n_owners_eligible'] >= 2).sum():,}")
skills.head()

copied skills with >= 2 eligible owners (RQ1): 67,061


,skill_id,file_sha,filename,location_class,frontmatter_valid,body_chars,has_scripts,has_references,sibling_count,n_repos,n_owners,n_repos_eligible,n_owners_eligible,n_languages
0,0,000041ef997179e0d940f1bc9e08bfccd5c00fd8,SKILL.md,other,0.0,214.0,0.0,0.0,1.0,2,2,0,0,0
1,1,000096e7cdc7bcd4a8031e889dd6021e7bc05fb7,SKILL.md,skills-dir,0.0,65.0,0.0,0.0,0.0,3,3,1,1,1
2,2,00009a13ac1585155030c88723c6661610031a56,SKILL.md,other,1.0,8179.0,0.0,0.0,1.0,2,2,0,0,0
3,3,0000acbe72917ea47293198e6f6a529cdda60603,SKILL.md,skills-dir,1.0,3200.0,0.0,0.0,0.0,2,2,0,0,0
4,4,0000d71485d176ceda25d2b28af218cfe754f794,SKILL.md,skills-dir,1.0,8434.0,0.0,1.0,5.0,18,10,16,8,2


## 6. Save

In [8]:
repo_cols = ["repo_id", "repo_full_name", "owner", "language", "language_kind",
             "stars", "forks", "is_fork", "created_at", "pushed_at",
             "n_skill_files", "n_distinct_skills", "is_aggregator", "eligible"]

repos[repo_cols].to_csv(DATA_DIR / "repos_clean.csv", index=False)
skills.to_csv(DATA_DIR / "skills_copied.csv", index=False)
(adoptions[["skill_id", "repo_id", "n_paths"]].sort_values(["skill_id", "repo_id"])
 .to_csv(DATA_DIR / "adoptions.csv", index=False))